# 빠른 시작: 첫 세션 실행

작은 Python 계산으로 **Session 생성 → 이벤트 스트리밍 → 완료 상태·최종 JSON·명령 결과 확인 → 자동 정리**를 경험합니다. 숫자 세 개만 입력하며 CSV·파일 다운로드는 이후 장에서 배웁니다. 세부 개념을 먼저 읽고 싶으면 [1장](chapters/01_concepts.ipynb)을 참고하세요.

OpenAI-hosted sandbox와 모델을 실제 사용하므로 비용이 발생합니다. API 키는 클라이언트에만 두고 요청·sandbox에 넣지 않습니다. 기본 RUN_LIVE=False는 API 준비 셀에서 멈춥니다. 키 없이 로컬 셀만 점검하려면 오프라인 검증기를 사용하세요.


<!-- colab-link -->
[독립 실행본 다운로드 안내](../docs/colab.md) · Colab 업로드용


**실습 범위:** 수동 실행은 API 호출·최종 응답·명령 이력·정리를 직접 보여줍니다. 정답 비교와 상세 실행 증거·receipt는 검증 스크립트가 실행 사본에만 추가합니다. [실습 안내와 검증 범위](../docs/validation.md)


## 실습 순서와 예상 출력

| 단계 | 직접 할 일 | 확인할 출력 |
| --- | --- | --- |
| 준비 | 키·모델을 설정하고 RUN_LIVE=True 선택 | 실행 허용·선택 모델 |
| 요청 | Agent·환경·작업 내용을 읽기 | 실행 환경·입력 설명 |
| 실행 | 실행 셀이 완료될 때까지 기다리기 | Session ID·진행 텍스트 |
| 결과 | 같은 셀에서 저장 상태·응답·명령 확인 | root Turn·JSON·명령 상태/종료 코드/stdout |
| 자동 정리 | 같은 셀의 finally 결과 확인 | Session 삭제·부재 확인 |

짧은 실행과 정리를 한 셀에 유지합니다. 결과가 보였다고 중단하지 말고 정리까지 완료되는지 확인하세요.


## 0. 의존성과 입력 준비

로컬에서는 터미널에서 uv sync --locked를 실행하고 프로젝트 .venv 커널을 선택하세요. 설치·커널 설정은 [환경 준비](setup.ipynb)를 참고하세요.

Colab에서는 [독립 실행본](../docs/colab.md)을 업로드하고 패키지·포함 파일 준비 셀을 실행한 뒤 Secrets를 설정합니다. GitHub 주소나 저장소 접근은 필요하지 않습니다. 아래 원본 bootstrap은 로컬에서 설치하지 않으며, Colab 독립 실행본에서는 직접 패키지를 설치하는 준비 코드로 바뀝니다.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")


## 1. 실제 실행 허용과 설정 확인

실제 호출은 아래 RUN_LIVE를 True로 바꾸고 이후 셀을 위에서 아래로 실행하세요. False에서 API 준비 셀이 오류 안내와 함께 멈추는 것은 의도한 동작입니다. 일반 Jupyter는 requires_api 태그만으로 셀을 건너뛰지 않습니다. .env.local의 RUN_LIVE 값은 실행 허용으로 사용하지 않습니다.

| 환경 | 키·모델 설정 |
| --- | --- |
| 로컬 | .env.local의 OPENAI_API_KEY·OPENAI_MODEL, 없으면 프로세스 환경변수 |
| Colab | Secrets의 OPENAI_API_KEY·OPENAI_MODEL |
| 모델 override | MODEL_OVERRIDE가 있으면 모델 설정보다 우선 |

키를 셀에 직접 입력하거나 출력하지 마세요. False에서는 키를 읽지 않으며 모델이 빈 문자열일 수 있습니다. True에서 필수 설정이 없으면 API 요청 전에 설정 오류로 멈춥니다.


In [ ]:
from tutorial_helpers.common import load_settings
from importlib.metadata import version

RUN_LIVE = False
MODEL_OVERRIDE = ""
settings = load_settings(RUN_LIVE, MODEL_OVERRIDE)
print({"api_enabled": settings.enabled, "sdk_version": version("openai"), "model": settings.model})


## 2. 최소 요청 구성

OpenAI가 관리하는 작은 sandbox에서 quickstart_sum.py를 작성해 숫자 세 개의 개수·합계를 출력하도록 요청합니다. 추가 패키지나 외부 서비스는 사용하지 않습니다.

| 요청 필드 | 의미 |
| --- | --- |
| agent | 모델과 Python 계산 지침 |
| environment | Python·명령을 실행할 OpenAI-hosted sandbox |
| input | 스크립트 작성·직접 실행·최종 JSON 반환 요청 |
| stream | 실행 중 이벤트 관찰 |

network.access=disabled는 sandbox의 외부 접근을 끕니다. 로컬 Python client의 OpenAI API 연결과 별개입니다. 모델과 sandbox 접근 권한도 별개일 수 있어 권한 오류가 나면 설정을 확인합니다. 다른 환경으로 자동 전환하지 않습니다.


In [ ]:
request = {
    "agent": {
        "model": settings.model or "CONFIGURE_OPENAI_MODEL_BEFORE_LIVE_RUN",
        "instructions": (
            "Use Python for the requested calculation. Do not delegate or use the web. "
            "Keep commentary brief. After running the script, return only the JSON "
            "object with count and total, without Markdown fences or explanation."
        ),
    },
    "environment": {
        "type": "openai_hosted",
        "container_size": "small",
        "network": {"access": "disabled"},
    },
    "input": (
        "Create /workspace/quickstart_sum.py using Python's standard library. "
        "Set numbers = [10, 20, 30] and print JSON with count=len(numbers) "
        "and total=sum(numbers). Run exactly: python /workspace/quickstart_sum.py. "
        "Inspect the actual stdout and exit code. Return that JSON as the final answer."
    ),
    "stream": True,
}
print("Execution environment:", request["environment"])


## 3. 실행과 자동 정리 준비

진행 텍스트·완료 상태·최종 JSON·명령 stdout은 서로 다른 확인 대상입니다. 완료 Turn이나 JSON 파싱 성공만으로 계산 정답을 보장하지 않습니다. 실제 원래 Turn의 명령과 상태·종료 코드·stdout을 표시하며 정확한 스크립트·정답 대조는 검증 도구가 수행합니다.

정리 helper는 실행 중이면 취소를 시도하고 삭제 충돌은 제한적으로 재시도한 뒤 Session 재조회가 404인지 확인합니다. 이는 API상 부재 확인이며 sandbox 물리 정리가 동기적으로 완료됐다는 뜻은 아닙니다.


In [ ]:
import json
from uuid import uuid4
from tutorial_helpers.common import cleanup_resources, new_session_metadata, check_event, safe_error


## 4. 생성·스트리밍·결과 확인·자동 정리

설정에서 RUN_LIVE=True를 선택하고 API 준비 셀을 실행한 뒤 상태 준비→실행 순서로 진행합니다. 상태 준비는 새 client로 한 번만 실행하세요. 다시 초기화하면 이전 Session 식별자를 잃을 수 있습니다. 생성 셀 재실행은 같은 실행의 중복 요청으로 차단합니다.

with_result_collection은 진행 이벤트와 최종 응답을 수집합니다. Session 생성 이벤트에서 ID를 즉시 저장하고, 수집된 결과의 Session ID도 보완합니다. 같은 원래 root Turn을 조회해 완료 상태를 확인한 뒤 JSON과 명령 이력을 출력합니다. requires_action은 이 기본 예제에서 자동 처리하지 않습니다.

실행·결과 확인·정리를 한 try/finally에 유지합니다. 스트림 종료만으로 작업이 취소되지는 않습니다. HTTP timeout=90, 상태·이력 조회 timeout=15를 사용하며 엄격한 전체 실행 시간 제한은 아닙니다. 중단하려면 Jupyter 인터럽트 후 정리 결과를 확인하세요. 강제 커널·프로세스 종료의 finally 실행은 보장하지 않습니다.


#### API 실행 허용 확인

설정 셀에서 `RUN_LIVE=True`를 선택한 뒤 이 준비 셀을 실행하세요. 기본 False에서는 여기서 오류 메시지와 함께 멈춥니다. 이후 API 셀에는 실행 여부 분기를 반복하지 않습니다. `requires_api` 태그는 오프라인 검증기용이며 Jupyter 자체의 접근 제어가 아닙니다.


In [ ]:
from openai import OpenAI

client = None
if not settings.enabled:
    raise RuntimeError("Set RUN_LIVE=True to run the API example. Use the validation script for offline checks.")
client = OpenAI(api_key=settings.api_key, timeout=90, max_retries=0)


In [ ]:
run_id = uuid4().hex
output_dir = settings.root / "outputs/quickstart" / run_id
output_dir.mkdir(parents=True, exist_ok=True)
sessions, agents, attempts = {}, {}, []
cleanup_errors = []
result = turn = None
commands = []
creation_attempted = False
results_checked = False
print("Local investigation records:", output_dir)


In [ ]:
if creation_attempted:
    raise RuntimeError("This run already attempted creation; prepare a fresh client and run state")
creation_attempted = True
try:
    # 1. Create the session and collect progress plus the final response.
    request["metadata"] = new_session_metadata("preface", run_id, attempts=attempts, output_dir=output_dir)
    with client.beta.agents.sessions.create(**request).with_result_collection() as stream:
        for event in stream:
            check_event(event)
            if event.type == "agent.session.created":
                sessions[event.session.id] = {"deleted": False}
                print("Session:", event.session.id, flush=True)
            elif event.type == "agent.session.turn.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "agent.session.requires_action":
                raise RuntimeError("Unexpected external action in this hosted example")
        result = stream.get_final_result()
    sessions.setdefault(result.session_id, {"deleted": False})

    # 2. Inspect the original completed root turn and parse its final answer.
    turn = client.beta.agents.sessions.turns.retrieve(
        result.turn_id, session_id=result.session_id, timeout=15,
    )
    if (turn.id != result.turn_id or turn.subagent_id is not None
        or turn.status != "completed" or result.turn.status != "completed"):
        raise RuntimeError("The original completed root turn was not confirmed")
    answer = json.loads(result.output_text)
    if not isinstance(answer, dict) or set(answer) != {"count", "total"} or any(type(v) is not int for v in answer.values()):
        raise RuntimeError("Expected a JSON object with integer count and total")
    print("\nRoot Turn:", turn.id, turn.status)
    print("Final JSON:", answer)

    # 3. Read command history before deleting the session.
    items = list(client.beta.agents.sessions.items.list(result.session_id, order="asc", limit=100, timeout=15))
    commands = [item for item in items if item.type == "command_execution" and item.turn_id == turn.id]
    if not commands:
        raise RuntimeError("The original turn has no saved command history")
    for command in commands:
        print("Command:", command.id, command.command)
        print("Status:", command.status, "Exit code:", command.exit_code)
        print("Command stdout/result:", command.output)
    results_checked = True
finally:
    # 4. Preserve the original error while attempting resource cleanup.
    original_error = sys.exc_info()[1]
    try:
        cleanup_errors[:] = cleanup_resources(client, sessions, agents, attempts)
    except Exception as cleanup_error:
        cleanup_errors.append(safe_error(cleanup_error))
    if cleanup_errors:
        if original_error is None:
            raise RuntimeError("Resource cleanup incomplete; inspect local records")
        original_error.add_note("Resource cleanup incomplete; inspect local records")
if not all(owned.get("verified_absent") for owned in sessions.values()):
    raise RuntimeError("Session absence was not confirmed")
print("Automatic session cleanup:", "verified")


## 5. 실행 종료 확인

아래는 메모리에 남은 결과와 정리 기록만 읽습니다. 이미 닫힌 client로 API를 호출하거나 Session을 새로 만들지 않습니다. 결과 확인 완료는 표시·형식 확인을 뜻하며 검증 도구의 정답 통과와 구분합니다.

usage=None은 미보고이며 숫자 0이 아닙니다. cached/reasoning 토큰은 각각 입력/출력의 부분집합이므로 합계에 다시 더하지 않습니다. 실행 셀에서 오류가 났다면 아래 요약보다 원래 오류와 cleanup 기록부터 확인하세요. 같은 실행을 다시 제출하지 말고 새 client·상태로 시작합니다.


In [ ]:
print("Root status:", turn.status if turn is not None else "not_completed")
print("Results inspected:", results_checked)
print("Command items displayed:", len(commands))
print("Usage (None means not reported):", turn.usage.to_dict() if turn is not None and turn.usage is not None else None)
print("Session cleanup:", sessions)
print("Cleanup errors:", cleanup_errors)


## 6. 실패·중단 후 다음 행동

| 상황 | 확인할 내용·다음 행동 |
| --- | --- |
| 설정 오류·401/403 | 키·모델·프로젝트 권한과 Agents API/sandbox 접근 확인, 키 출력 금지 |
| Turn 실패·취소·예상하지 않은 외부 작업 | 성공으로 처리하지 않고 원래 오류·정리 기록 확인 |
| JSON 형식 오류·명령 이력 누락 | 완료 문구만 믿지 않고 해당 Turn 이력 확인 |
| 삭제 충돌·정리 실패 | 제한적 재시도 결과와 creation.json의 이번 실행 소유 자원 확인 |
| 연결 종료·timeout | 원래 작업의 상태부터 조사, 같은 입력을 새 Session에 무조건 재전송하지 않음 |
| 커널 강제 종료 | outputs/quickstart/<run_id>/creation.json의 고유 metadata·저장 ID로 해당 실행만 정리 |

이 예제는 오류 시에도 정리를 시도하며 client를 닫습니다. 복구 조회가 필요하면 새 client를 사용하고 먼저 Session이 이미 삭제됐는지 확인하세요. ID를 받지 못했다면 creation.json의 tutorial_run_id와 tutorial_creation_id로 소유 자원을 식별합니다. 자동 재개는 구현하지 않습니다.

```python
# Optional manual recovery: create a fresh client and verify ownership first.
# from openai import OpenAI
# from tutorial_helpers.common import load_settings, delete_session_safely
# recovery_settings = load_settings(run_live=True)
# with OpenAI(api_key=recovery_settings.api_key, timeout=15, max_retries=0) as recovery_client:
#     delete_session_safely(recovery_client, "saved_session_id_here")
```

## 선택 조사와 다음 단계

진행을 더 조사할 때만 [Dashboard 실행 기록](https://platform.openai.com/logs?api=agents)에서 표시된 Session ID를 찾아보세요. 이 노트북은 Trace API·export를 호출하지 않습니다. UI 표시를 자동 검증했다고 가정하지 않습니다.

- [1장 핵심 개념](chapters/01_concepts.ipynb): 요청과 실행 주체를 정리합니다.
- [2장 CSV 분석](chapters/02_first_agent.ipynb): 같은 흐름을 데이터 분석에 적용합니다.
- [14장 관찰 연결 복구](chapters/14_operations.ipynb): 재연결과 입력 재전송을 구분합니다.

수동 실행의 creation.json은 소유 자원 조사 기록입니다. 자동 검증은 uv run --group dev python scripts/run_quickstart.py --live로 실행하며 실행 사본에만 정답/명령 검사와 summary.json을 추가합니다. 키 없이 로컬 셀을 점검하려면 uv run --group dev python scripts/check_notebooks.py --execute를 사용하세요. 검증 사본은 outputs/quickstart-validation/quickstart.executed.ipynb, 검증 기록은 outputs/quickstart/<run_id>/summary.json에 저장합니다. 로컬 파일 삭제는 원격 Session 삭제와 다릅니다.


## 공식 참고 자료

### Agents API 전용 문서

- [Agents API quickstart](https://developers.openai.com/api/docs/guides/agents-api/quickstart) — 세션 생성·스트리밍·결과 확인·자원 정리.
- [Events and items](https://developers.openai.com/api/docs/guides/agents-api/sessions/events) — 이벤트 스트림·완료 상태·저장된 items와 turns 조회.
- [Manage sessions](https://developers.openai.com/api/docs/guides/agents-api/sessions/manage) — 현재 상태·pending action 확인과 세션 삭제.
- [OpenAI-hosted sandboxes](https://developers.openai.com/api/docs/guides/agents-api/environments/openai-hosted) — packages·setup_commands·초기 파일·네트워크 설정.

링크 확인: 2026-11-03. 문서 링크 확인은 API 실습 실행 검증과 별개입니다.
